# Olist Data Quality Exploration using SQL - WED 12052026

This notebook uses SQL primarily to explore the following dataset.
**Dataset Source**: Olist
**Bronze Tables listed after running**: `%fs ls /Volumes/bh2026-winford-uc-dev/bricks_hub_olist/olist`

## 0. Initialization & Setup

In [0]:
USE CATALOG `bh2026-winford-uc-dev`;
USE SCHEMA bricks_hub_olist;
SELECT current_catalog() AS current_catalog, current_schema() AS current_schema;

In [0]:
%fs ls /Volumes/bh2026-winford-uc-dev/bricks_hub_olist/olist

## 1. Load and Display Data

In [0]:
-- Display basic info for the olist bronze_orders data table
SELECT 
  (SELECT COUNT(*) FROM bronze_orders) AS total_rows,
  (SELECT COUNT(*) FROM information_schema.columns
   WHERE table_catalog = 'bh2026-winford-uc-dev'
     AND table_schema = 'bricks_hub_olist'
     AND table_name = 'bronze_orders') AS total_columns,
  (SELECT ARRAY_JOIN(ARRAY_AGG(column_name), ', ') FROM information_schema.columns
   WHERE table_catalog = 'bh2026-winford-uc-dev'
     AND table_schema = 'bricks_hub_olist'
     AND table_name = 'bronze_orders') AS column_names;

## 2. View First 5 Rows

In [0]:
-- View first 5 rows of the dataset
SELECT * FROM bronze_orders LIMIT 5;

## 3. Schema and Data Types

In [0]:
-- Schema and data types
DESCRIBE TABLE bronze_orders;

## 4. Statistical Summary

In [0]:
-- Statistical summary for bronze_orders (based on DESCRIBE results)
-- Columns: order_id (string), customer_id (string), order_status (string),
--          order_purchase_timestamp (timestamp), order_approved_at (timestamp),
--          order_delivered_carrier_date (timestamp), order_delivered_customer_date (timestamp),
--          order_estimated_delivery_date (timestamp), _rescued_data (string)

SELECT 'count' AS summary,
  CAST(COUNT(order_id) AS STRING) AS order_id,
  CAST(COUNT(customer_id) AS STRING) AS customer_id,
  CAST(COUNT(order_status) AS STRING) AS order_status,
  CAST(COUNT(order_purchase_timestamp) AS STRING) AS order_purchase_timestamp,
  CAST(COUNT(order_approved_at) AS STRING) AS order_approved_at,
  CAST(COUNT(order_delivered_carrier_date) AS STRING) AS order_delivered_carrier_date,
  CAST(COUNT(order_delivered_customer_date) AS STRING) AS order_delivered_customer_date,
  CAST(COUNT(order_estimated_delivery_date) AS STRING) AS order_estimated_delivery_date
FROM bronze_orders

UNION ALL

SELECT 'nulls',
  CAST(SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS STRING),
  CAST(SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS STRING),
  CAST(SUM(CASE WHEN order_status IS NULL THEN 1 ELSE 0 END) AS STRING),
  CAST(SUM(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END) AS STRING),
  CAST(SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS STRING),
  CAST(SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) AS STRING),
  CAST(SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS STRING),
  CAST(SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END) AS STRING)
FROM bronze_orders

UNION ALL

SELECT 'distinct',
  CAST(COUNT(DISTINCT order_id) AS STRING),
  CAST(COUNT(DISTINCT customer_id) AS STRING),
  CAST(COUNT(DISTINCT order_status) AS STRING),
  CAST(COUNT(DISTINCT order_purchase_timestamp) AS STRING),
  CAST(COUNT(DISTINCT order_approved_at) AS STRING),
  CAST(COUNT(DISTINCT order_delivered_carrier_date) AS STRING),
  CAST(COUNT(DISTINCT order_delivered_customer_date) AS STRING),
  CAST(COUNT(DISTINCT order_estimated_delivery_date) AS STRING)
FROM bronze_orders

UNION ALL

SELECT 'min',
  MIN(order_id),
  MIN(customer_id),
  MIN(order_status),
  CAST(MIN(order_purchase_timestamp) AS STRING),
  CAST(MIN(order_approved_at) AS STRING),
  CAST(MIN(order_delivered_carrier_date) AS STRING),
  CAST(MIN(order_delivered_customer_date) AS STRING),
  CAST(MIN(order_estimated_delivery_date) AS STRING)
FROM bronze_orders

UNION ALL

SELECT 'max',
  MAX(order_id),
  MAX(customer_id),
  MAX(order_status),
  CAST(MAX(order_purchase_timestamp) AS STRING),
  CAST(MAX(order_approved_at) AS STRING),
  CAST(MAX(order_delivered_carrier_date) AS STRING),
  CAST(MAX(order_delivered_customer_date) AS STRING),
  CAST(MAX(order_estimated_delivery_date) AS STRING)
FROM bronze_orders;

## 5. Data Quality Check - Null/Missing Values

In [0]:
-- Check for null values in each column
SELECT
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulls,
  SUM(CASE WHEN order_status IS NULL THEN 1 ELSE 0 END) AS order_status_nulls,
  SUM(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END) AS order_purchase_timestamp_nulls,
  SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS order_approved_at_nulls,
  SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) AS order_delivered_carrier_date_nulls,
  SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS order_delivered_customer_date_nulls,
  SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END) AS order_estimated_delivery_date_nulls
FROM bronze_orders;

## 6. Unique Values Count

In [0]:
-- Count unique values for each column
SELECT
  COUNT(DISTINCT order_id) AS order_id_unique,
  COUNT(DISTINCT customer_id) AS customer_id_unique,
  COUNT(DISTINCT order_status) AS order_status_unique,
  COUNT(DISTINCT order_purchase_timestamp) AS order_purchase_timestamp_unique,
  COUNT(DISTINCT order_approved_at) AS order_approved_at_unique,
  COUNT(DISTINCT order_delivered_carrier_date) AS order_delivered_carrier_date_unique,
  COUNT(DISTINCT order_delivered_customer_date) AS order_delivered_customer_date_unique,
  COUNT(DISTINCT order_estimated_delivery_date) AS order_estimated_delivery_date_unique
FROM bronze_orders;

   
## 7. Distribution by Order Status

In [0]:
-- Show distribution by order status
SELECT order_status, COUNT(*) AS count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM bronze_orders), 2) AS pct
FROM bronze_orders
GROUP BY order_status
ORDER BY count DESC;

   
## 8. Timestamp Coverage Analysis

In [0]:
-- Analyze timestamp coverage and date ranges
SELECT
  'order_purchase_timestamp' AS column_name,
  COUNT(order_purchase_timestamp) AS non_null_count,
  CAST(MIN(order_purchase_timestamp) AS STRING) AS min_value,
  CAST(MAX(order_purchase_timestamp) AS STRING) AS max_value
FROM bronze_orders
UNION ALL
SELECT 'order_approved_at',
  COUNT(order_approved_at),
  CAST(MIN(order_approved_at) AS STRING),
  CAST(MAX(order_approved_at) AS STRING)
FROM bronze_orders
UNION ALL
SELECT 'order_delivered_carrier_date',
  COUNT(order_delivered_carrier_date),
  CAST(MIN(order_delivered_carrier_date) AS STRING),
  CAST(MAX(order_delivered_carrier_date) AS STRING)
FROM bronze_orders
UNION ALL
SELECT 'order_delivered_customer_date',
  COUNT(order_delivered_customer_date),
  CAST(MIN(order_delivered_customer_date) AS STRING),
  CAST(MAX(order_delivered_customer_date) AS STRING)
FROM bronze_orders
UNION ALL
SELECT 'order_estimated_delivery_date',
  COUNT(order_estimated_delivery_date),
  CAST(MIN(order_estimated_delivery_date) AS STRING),
  CAST(MAX(order_estimated_delivery_date) AS STRING)
FROM bronze_orders;

   
## 9. Delivery Time Analysis

In [0]:
-- Delivery time analysis (days between purchase and delivery)
SELECT
  COUNT(*) AS delivered_orders,
  ROUND(AVG(DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)), 1) AS avg_delivery_days,
  MIN(DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)) AS min_delivery_days,
  MAX(DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)) AS max_delivery_days,
  ROUND(STDDEV(DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)), 1) AS stddev_delivery_days
FROM bronze_orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_purchase_timestamp IS NOT NULL;

   
## 10. Customer Order Analysis

In [0]:
-- Top 10 customers by order count
SELECT customer_id,
  COUNT(order_id) AS total_orders
FROM bronze_orders
GROUP BY customer_id
ORDER BY total_orders DESC
LIMIT 10;

   
## 11. Order Status Breakdown Over Time

In [0]:
-- Order status breakdown by month
SELECT
  DATE_TRUNC('month', order_purchase_timestamp) AS order_month,
  order_status,
  COUNT(*) AS order_count
FROM bronze_orders
WHERE order_purchase_timestamp IS NOT NULL
GROUP BY order_month, order_status
ORDER BY order_month DESC, order_count DESC
LIMIT 20;

   
## 12. Delivery Performance Analysis

In [0]:
-- Delivery performance: on-time vs late (compared to estimated delivery date)
SELECT
  COUNT(*) AS total_delivered,
  SUM(CASE WHEN order_delivered_customer_date <= order_estimated_delivery_date THEN 1 ELSE 0 END) AS on_time,
  SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END) AS late,
  ROUND(SUM(CASE WHEN order_delivered_customer_date <= order_estimated_delivery_date THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS on_time_pct,
  ROUND(AVG(DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date)), 1) AS avg_days_vs_estimate
FROM bronze_orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;

   
## 13. Approval Time Analysis

In [0]:
-- Time between purchase and approval (hours)
SELECT
  COUNT(*) AS approved_orders,
  ROUND(AVG((UNIX_TIMESTAMP(order_approved_at) - UNIX_TIMESTAMP(order_purchase_timestamp)) / 3600), 1) AS avg_approval_hours,
  ROUND(MIN((UNIX_TIMESTAMP(order_approved_at) - UNIX_TIMESTAMP(order_purchase_timestamp)) / 3600), 1) AS min_approval_hours,
  ROUND(MAX((UNIX_TIMESTAMP(order_approved_at) - UNIX_TIMESTAMP(order_purchase_timestamp)) / 3600), 1) AS max_approval_hours
FROM bronze_orders
WHERE order_approved_at IS NOT NULL
  AND order_purchase_timestamp IS NOT NULL;

   
## 14. Order Date Range

In [0]:
-- Order date range
SELECT
  MIN(order_purchase_timestamp) AS earliest_order,
  MAX(order_purchase_timestamp) AS latest_order,
  DATEDIFF(MAX(order_purchase_timestamp), MIN(order_purchase_timestamp)) AS date_span_days
FROM bronze_orders;

   
## 15. Sample Data for Review

In [0]:
-- Sample 20 rows for review
SELECT * FROM bronze_orders LIMIT 20;

# DATA QUALITY CHECKS - COMPREHENSIVE VALIDATION

## 16. Null or Missing Value Checks

In [0]:
-- Comprehensive null and missing value analysis
SELECT
  column_name,
  null_count,
  total_count,
  ROUND((null_count / total_count) * 100, 2) AS pct_null
FROM (
  SELECT 'order_id' AS column_name, SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_count, COUNT(*) AS total_count FROM bronze_orders
  UNION ALL
  SELECT 'customer_id', SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM bronze_orders
  UNION ALL
  SELECT 'order_status', SUM(CASE WHEN order_status IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM bronze_orders
  UNION ALL
  SELECT 'order_purchase_timestamp', SUM(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM bronze_orders
  UNION ALL
  SELECT 'order_approved_at', SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM bronze_orders
  UNION ALL
  SELECT 'order_delivered_carrier_date', SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM bronze_orders
  UNION ALL
  SELECT 'order_delivered_customer_date', SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM bronze_orders
  UNION ALL
  SELECT 'order_estimated_delivery_date', SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM bronze_orders
)
ORDER BY pct_null DESC;

## 17. Primary Key Uniqueness Validation

In [0]:
-- Primary key uniqueness validation (order_id should be unique)
WITH totals AS (
  SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT order_id) AS unique_order_ids
  FROM bronze_orders
),
duplicates AS (
  SELECT order_id, COUNT(*) AS cnt
  FROM bronze_orders
  GROUP BY order_id
  HAVING COUNT(*) > 1
)
SELECT
  t.total_records,
  t.unique_order_ids,
  t.total_records - t.unique_order_ids AS duplicate_keys,
  (SELECT COUNT(*) FROM duplicates) AS duplicate_key_groups,
  (SELECT MAX(cnt) FROM duplicates) AS max_duplicates_per_key
FROM totals t;

## 18. Duplicate Record Detection

In [0]:
-- Detect completely duplicate rows
SELECT order_id, customer_id, order_status, order_purchase_timestamp,
  order_approved_at, order_delivered_carrier_date, order_delivered_customer_date,
  order_estimated_delivery_date,
  COUNT(*) AS duplicate_count
FROM bronze_orders
GROUP BY order_id, customer_id, order_status, order_purchase_timestamp,
  order_approved_at, order_delivered_carrier_date, order_delivered_customer_date,
  order_estimated_delivery_date
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 10;

## 19. Referential Integrity Validation

In [0]:
-- Referential integrity checks
-- Check for orders missing key identifiers
SELECT
  (SELECT COUNT(*) FROM bronze_orders WHERE customer_id IS NULL) AS orders_without_customer,
  (SELECT COUNT(*) FROM bronze_orders WHERE order_id IS NULL) AS orders_without_order_id,
  (SELECT COUNT(*) FROM bronze_orders WHERE order_status IS NULL) AS orders_without_status,
  (SELECT COUNT(*) FROM bronze_orders WHERE order_purchase_timestamp IS NULL) AS orders_without_purchase_date;

## 20. Data Type Validation

In [0]:
-- Data type validation - compare actual vs expected types
SELECT
  column_name,
  data_type AS actual_type,
  CASE column_name
    WHEN 'order_id' THEN 'string'
    WHEN 'customer_id' THEN 'string'
    WHEN 'order_status' THEN 'string'
    WHEN 'order_purchase_timestamp' THEN 'timestamp'
    WHEN 'order_approved_at' THEN 'timestamp'
    WHEN 'order_delivered_carrier_date' THEN 'timestamp'
    WHEN 'order_delivered_customer_date' THEN 'timestamp'
    WHEN 'order_estimated_delivery_date' THEN 'timestamp'
    WHEN '_rescued_data' THEN 'string'
  END AS expected_type,
  CASE WHEN data_type = CASE column_name
    WHEN 'order_id' THEN 'string'
    WHEN 'customer_id' THEN 'string'
    WHEN 'order_status' THEN 'string'
    WHEN 'order_purchase_timestamp' THEN 'timestamp'
    WHEN 'order_approved_at' THEN 'timestamp'
    WHEN 'order_delivered_carrier_date' THEN 'timestamp'
    WHEN 'order_delivered_customer_date' THEN 'timestamp'
    WHEN 'order_estimated_delivery_date' THEN 'timestamp'
    WHEN '_rescued_data' THEN 'string'
  END THEN '✓ Match' ELSE '✗ Drift' END AS status
FROM information_schema.columns
WHERE table_catalog = 'bh2026-winford-uc-dev'
  AND table_schema = 'bricks_hub_olist'
  AND table_name = 'bronze_orders';

   
## 21. Timestamp Range Validation

In [0]:
-- Timestamp range validation (check for impossible/future dates)
SELECT
  SUM(CASE WHEN order_purchase_timestamp > CURRENT_TIMESTAMP() THEN 1 ELSE 0 END) AS future_purchase_dates,
  SUM(CASE WHEN order_approved_at > CURRENT_TIMESTAMP() THEN 1 ELSE 0 END) AS future_approval_dates,
  SUM(CASE WHEN order_delivered_customer_date > CURRENT_TIMESTAMP() THEN 1 ELSE 0 END) AS future_delivery_dates,
  SUM(CASE WHEN order_purchase_timestamp < '2015-01-01' THEN 1 ELSE 0 END) AS purchase_before_2015,
  SUM(CASE WHEN order_delivered_customer_date < order_purchase_timestamp THEN 1 ELSE 0 END) AS delivered_before_purchased
FROM bronze_orders;

## 22. String Length & Pattern Checks

In [0]:
-- String length and pattern validation
SELECT
  'order_id' AS column_name,
  MIN(LENGTH(order_id)) AS min_length,
  MAX(LENGTH(order_id)) AS max_length,
  ROUND(AVG(LENGTH(order_id)), 2) AS avg_length
FROM bronze_orders
UNION ALL
SELECT 'customer_id',
  MIN(LENGTH(customer_id)),
  MAX(LENGTH(customer_id)),
  ROUND(AVG(LENGTH(customer_id)), 2)
FROM bronze_orders
UNION ALL
SELECT 'order_status',
  MIN(LENGTH(order_status)),
  MAX(LENGTH(order_status)),
  ROUND(AVG(LENGTH(order_status)), 2)
FROM bronze_orders;

## 23. Allowed Value / Domain Validation

In [0]:
-- Domain value validation - order_status should only contain valid values
SELECT
  (SELECT COUNT(DISTINCT order_status) FROM bronze_orders) AS total_unique_statuses,
  order_status,
  COUNT(*) AS count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM bronze_orders), 2) AS pct
FROM bronze_orders
GROUP BY order_status
ORDER BY count DESC;

## 24. Business Rule Consistency

In [0]:
-- Business rule consistency checks
SELECT
  -- Orders delivered but never approved
  SUM(CASE WHEN order_delivered_customer_date IS NOT NULL AND order_approved_at IS NULL THEN 1 ELSE 0 END) AS delivered_without_approval,
  -- Orders delivered to customer before carrier pickup
  SUM(CASE WHEN order_delivered_customer_date < order_delivered_carrier_date THEN 1 ELSE 0 END) AS delivered_before_carrier,
  -- Orders approved before purchase
  SUM(CASE WHEN order_approved_at < order_purchase_timestamp THEN 1 ELSE 0 END) AS approved_before_purchase,
  -- Orders with status 'delivered' but no delivery date
  SUM(CASE WHEN order_status = 'delivered' AND order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS status_delivered_no_date,
  -- Orders with delivery date but status not 'delivered'
  SUM(CASE WHEN order_delivered_customer_date IS NOT NULL AND order_status != 'delivered' THEN 1 ELSE 0 END) AS has_date_not_delivered
FROM bronze_orders;

## 25. Cross-Column Consistency

In [0]:
-- Cross-column consistency checks
SELECT
  -- Orders where timestamps are out of logical sequence
  SUM(CASE WHEN order_approved_at < order_purchase_timestamp THEN 1 ELSE 0 END) AS approval_before_purchase,
  SUM(CASE WHEN order_delivered_carrier_date < order_approved_at THEN 1 ELSE 0 END) AS carrier_before_approval,
  SUM(CASE WHEN order_delivered_customer_date < order_delivered_carrier_date THEN 1 ELSE 0 END) AS customer_before_carrier,
  -- Orders with estimated delivery before purchase
  SUM(CASE WHEN order_estimated_delivery_date < order_purchase_timestamp THEN 1 ELSE 0 END) AS estimate_before_purchase
FROM bronze_orders
WHERE order_purchase_timestamp IS NOT NULL;

## 26. Timeliness / Freshness Checks

In [0]:
-- Data freshness/timeliness checks
SELECT
  MIN(order_purchase_timestamp) AS earliest_order,
  MAX(order_purchase_timestamp) AS latest_order,
  DATEDIFF(CURRENT_DATE(), MAX(order_purchase_timestamp)) AS days_since_last_order,
  SUM(CASE WHEN order_purchase_timestamp > CURRENT_TIMESTAMP() THEN 1 ELSE 0 END) AS future_orders
FROM bronze_orders;

## 27. Completeness Check

In [0]:
-- Completeness check - percentage of non-null values per column
SELECT
  ROUND(COUNT(order_id) / COUNT(*) * 100, 2) AS order_id_pct,
  ROUND(COUNT(customer_id) / COUNT(*) * 100, 2) AS customer_id_pct,
  ROUND(COUNT(order_status) / COUNT(*) * 100, 2) AS order_status_pct,
  ROUND(COUNT(order_purchase_timestamp) / COUNT(*) * 100, 2) AS purchase_ts_pct,
  ROUND(COUNT(order_approved_at) / COUNT(*) * 100, 2) AS approved_at_pct,
  ROUND(COUNT(order_delivered_carrier_date) / COUNT(*) * 100, 2) AS carrier_date_pct,
  ROUND(COUNT(order_delivered_customer_date) / COUNT(*) * 100, 2) AS customer_date_pct,
  ROUND(COUNT(order_estimated_delivery_date) / COUNT(*) * 100, 2) AS estimated_date_pct
FROM bronze_orders;

## 28. Volume Check Against Historical Data

In [0]:
-- Volume analysis - daily order distribution
SELECT
  CAST(order_purchase_timestamp AS DATE) AS order_day,
  COUNT(*) AS record_count
FROM bronze_orders
WHERE order_purchase_timestamp IS NOT NULL
GROUP BY CAST(order_purchase_timestamp AS DATE)
ORDER BY order_day;

   
## 29. Statistical Distribution Checks

In [0]:
-- Statistical distribution analysis of delivery times (in days)
WITH delivery_times AS (
  SELECT
    DATEDIFF(order_delivered_customer_date, order_purchase_timestamp) AS delivery_days
  FROM bronze_orders
  WHERE order_delivered_customer_date IS NOT NULL
    AND order_purchase_timestamp IS NOT NULL
)
SELECT
  'delivery_days' AS metric,
  PERCENTILE_APPROX(delivery_days, 0.25) AS Q1,
  PERCENTILE_APPROX(delivery_days, 0.50) AS Median,
  PERCENTILE_APPROX(delivery_days, 0.75) AS Q3,
  ROUND(SKEWNESS(delivery_days), 4) AS Skewness,
  ROUND(KURTOSIS(delivery_days), 4) AS Kurtosis
FROM delivery_times;

## 30. Outlier Detection

In [0]:
-- Outlier detection using IQR method on delivery times (days)
WITH delivery_times AS (
  SELECT
    order_id,
    DATEDIFF(order_delivered_customer_date, order_purchase_timestamp) AS delivery_days
  FROM bronze_orders
  WHERE order_delivered_customer_date IS NOT NULL
    AND order_purchase_timestamp IS NOT NULL
),
quartiles AS (
  SELECT
    PERCENTILE_APPROX(delivery_days, 0.25) AS Q1,
    PERCENTILE_APPROX(delivery_days, 0.75) AS Q3
  FROM delivery_times
)
SELECT
  q.Q1,
  q.Q3,
  q.Q3 - q.Q1 AS IQR,
  q.Q1 - 1.5 * (q.Q3 - q.Q1) AS lower_fence,
  q.Q3 + 1.5 * (q.Q3 - q.Q1) AS upper_fence,
  SUM(CASE WHEN dt.delivery_days < q.Q1 - 1.5 * (q.Q3 - q.Q1) THEN 1 ELSE 0 END) AS lower_outliers,
  SUM(CASE WHEN dt.delivery_days > q.Q3 + 1.5 * (q.Q3 - q.Q1) THEN 1 ELSE 0 END) AS upper_outliers
FROM delivery_times dt CROSS JOIN quartiles q
GROUP BY q.Q1, q.Q3;

## 31. Schema Drift Detection

In [0]:
-- Schema drift detection - compare actual vs expected column types
SELECT
  column_name,
  data_type AS actual_type,
  CASE column_name
    WHEN 'order_id' THEN 'string'
    WHEN 'customer_id' THEN 'string'
    WHEN 'order_status' THEN 'string'
    WHEN 'order_purchase_timestamp' THEN 'timestamp'
    WHEN 'order_approved_at' THEN 'timestamp'
    WHEN 'order_delivered_carrier_date' THEN 'timestamp'
    WHEN 'order_delivered_customer_date' THEN 'timestamp'
    WHEN 'order_estimated_delivery_date' THEN 'timestamp'
    WHEN '_rescued_data' THEN 'string'
  END AS expected_type,
  CASE WHEN data_type = CASE column_name
    WHEN 'order_id' THEN 'string'
    WHEN 'customer_id' THEN 'string'
    WHEN 'order_status' THEN 'string'
    WHEN 'order_purchase_timestamp' THEN 'timestamp'
    WHEN 'order_approved_at' THEN 'timestamp'
    WHEN 'order_delivered_carrier_date' THEN 'timestamp'
    WHEN 'order_delivered_customer_date' THEN 'timestamp'
    WHEN 'order_estimated_delivery_date' THEN 'timestamp'
    WHEN '_rescued_data' THEN 'string'
  END THEN 'Match' ELSE 'DRIFT DETECTED' END AS status
FROM information_schema.columns
WHERE table_catalog = 'bh2026-winford-uc-dev'
  AND table_schema = 'bricks_hub_olist'
  AND table_name = 'bronze_orders';

## 32. Duplicate File Ingestion Check

In [0]:
-- Duplicate file ingestion check
WITH counts AS (
  SELECT COUNT(*) AS total_rows FROM bronze_orders
),
distinct_counts AS (
  SELECT COUNT(*) AS distinct_rows FROM (SELECT DISTINCT * FROM bronze_orders)
)
SELECT
  c.total_rows,
  d.distinct_rows,
  c.total_rows - d.distinct_rows AS exact_duplicate_rows,
  ROUND((c.total_rows - d.distinct_rows) / c.total_rows * 100, 2) AS duplication_rate_pct
FROM counts c CROSS JOIN distinct_counts d;

   
## 33. Negative / Invalid Value Checks

In [0]:
-- Negative / invalid value checks (timestamp logic violations)
SELECT
  SUM(CASE WHEN order_delivered_customer_date < order_purchase_timestamp THEN 1 ELSE 0 END) AS delivery_before_purchase,
  SUM(CASE WHEN order_approved_at < order_purchase_timestamp THEN 1 ELSE 0 END) AS approval_before_purchase,
  SUM(CASE WHEN order_delivered_carrier_date < order_approved_at THEN 1 ELSE 0 END) AS carrier_before_approval,
  SUM(CASE WHEN DATEDIFF(order_delivered_customer_date, order_purchase_timestamp) > 365 THEN 1 ELSE 0 END) AS delivery_over_1_year
FROM bronze_orders;

## 34. Percentage / Total Consistency Check

In [0]:
-- Percentage / total consistency check
-- Verify that order counts by status sum to total
WITH total AS (
  SELECT COUNT(*) AS total_orders FROM bronze_orders
),
by_status AS (
  SELECT order_status, COUNT(*) AS status_count
  FROM bronze_orders
  GROUP BY order_status
)
SELECT
  t.total_orders,
  (SELECT SUM(status_count) FROM by_status) AS sum_of_status_counts,
  CASE WHEN t.total_orders = (SELECT SUM(status_count) FROM by_status)
    THEN 'Match' ELSE 'Mismatch' END AS consistency_check
FROM total t;

## 35. Hierarchy Validation

In [0]:
-- Hierarchy validation
SELECT
  -- Customers appearing with multiple order statuses (expected - valid)
  (SELECT COUNT(*) FROM (
    SELECT customer_id FROM bronze_orders GROUP BY customer_id HAVING COUNT(DISTINCT order_status) > 1
  )) AS customers_multi_status,
  -- Min/max/avg orders per customer
  (SELECT MIN(cnt) FROM (SELECT COUNT(*) AS cnt FROM bronze_orders GROUP BY customer_id)) AS min_orders_per_customer,
  (SELECT MAX(cnt) FROM (SELECT COUNT(*) AS cnt FROM bronze_orders GROUP BY customer_id)) AS max_orders_per_customer,
  (SELECT ROUND(AVG(cnt), 2) FROM (SELECT COUNT(*) AS cnt FROM bronze_orders GROUP BY customer_id)) AS avg_orders_per_customer;

## 36. Audit Column Consistency

In [0]:
-- Audit column consistency - validate timestamp sequencing
-- Expected sequence: purchase → approved → carrier → customer delivery
SELECT
  COUNT(*) AS total_records,
  SUM(CASE WHEN order_approved_at >= order_purchase_timestamp THEN 1 ELSE 0 END) AS valid_approval_sequence,
  SUM(CASE WHEN order_delivered_carrier_date >= order_approved_at THEN 1 ELSE 0 END) AS valid_carrier_sequence,
  SUM(CASE WHEN order_delivered_customer_date >= order_delivered_carrier_date THEN 1 ELSE 0 END) AS valid_delivery_sequence,
  SUM(CASE 
    WHEN order_approved_at >= order_purchase_timestamp
     AND order_delivered_carrier_date >= order_approved_at
     AND order_delivered_customer_date >= order_delivered_carrier_date
    THEN 1 ELSE 0 END) AS fully_valid_sequence
FROM bronze_orders
WHERE order_delivered_customer_date IS NOT NULL;

In [0]:
-- Table history for audit trail
DESCRIBE HISTORY bronze_orders;